# TAPOCB-NN: eight separate checkpoints

The 11-feature, two-hidden-layer `PolygonAlgNN` is preserved. Each stationary
preference scenario has a separate checkpoint; the scenario is **not** a DNN
input. Train/test boundaries group every augmented polygon with its original.
Use labels obtained from cached TAPOCT triplets; no routing solver is rerun.

Run from the directory containing `tapoct_fast.py` and
`TAPOCT_robustness_cell.py`, after the TAPOCT cache has finished writing.


In [12]:
import copy
import json
import os
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

import tapoct_fast as tf
import TAPOCT_robustness_cell as rb

DATA = {
    'properties': 'global_properties_osrm_seconds.pkl',
    'map': 'AUG_global_map_osrm_seconds.pkl',
    'results': 'epos_cache_fd.pkl',
    'couriers': 'courier_training_checkpoint_fd_seconds.pkl',
    'preferences': 'glob_pref_map.pkl',
}
TRAINING_SEED = 42  # one replicated scenario sample; change and retrain to test seed sensitivity
OUT_DIR = Path('courier_models_fd_8_scenarios')
OUT_DIR.mkdir(exist_ok=True)
with open(DATA['properties'], 'rb') as f: properties_global = pickle.load(f)
with open(DATA['map'], 'rb') as f: AUG_global_map = pickle.load(f)
with open(DATA['results'], 'rb') as f: EPOS_results = pickle.load(f)
with open(DATA['couriers'], 'rb') as f: trainer = pickle.load(f)
with open(DATA['preferences'], 'rb') as f: GLOBAL_PREFERENCES_MAP = pickle.load(f)

cfg = tf.TAPOCTFastConfig(n_passes_min=2, n_passes_max=3,
                           label_mode='omd', max_gap=0.01, B=43200.0, seed=42)
assert tuple(cfg.algorithms) == ('MatNet', 'NLNS', 'lkh3', 'Christofides')
print('Loaded', len(EPOS_results), 'polygon records and', len(trainer), 'couriers')


Loaded 3850 polygon records and 1 couriers


In [13]:
# Generate eight soft-label maps from the same saved solver calls.
# Spatial/Gaussian uses the restored preference map; independent/Gaussian
# permutes its vectors across polygons to remove the cluster association.
# For the spatial/Gaussian reference, train against the actual courier policies.
# The seven remaining sensitivity scenarios use recomputed fixed-triplet labels.
maps, labels, cluster_summary, cluster_params, diagnostics = (
    rb.run_distribution_robustness(
        EPOS_results, AUG_global_map, properties_global, cfg,
        seeds=(TRAINING_SEED,), gaussian_reference=GLOBAL_PREFERENCES_MAP,
    )
)
scenario_labels = labels[TRAINING_SEED]
cluster_summary.to_csv(OUT_DIR / 'cluster_summary_train_seed.csv', index=False)
print('Scenarios:', [(m, d, len(scenario_labels[m][d]))
                     for m in rb.SPATIAL_MODES for d in rb.SCENARIOS])


Scenarios: [('independent', 'gaussian', 3850), ('independent', 'beta', 3850), ('independent', 'dirichlet', 3850), ('independent', 'ordinal_posterior', 3850), ('spatial', 'gaussian', 3850), ('spatial', 'beta', 3850), ('spatial', 'dirichlet', 3850), ('spatial', 'ordinal_posterior', 3850)]


In [14]:
class PolygonAlgNN(nn.Module):
    
    def __init__(self, input_dim=11, hidden_dim=128, classes=4):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(0.15),
            
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(0.15),
            
            nn.Linear(hidden_dim, classes),
        )
    
    def forward(self, x):
        return self.network(x)



In [15]:
FEATURE_NAMES = (
    'area_km2', 'centroid_lat', 'centroid_lon', 'box_width_km',
    'box_height_km', 'normalized_lat', 'normalized_lon',
    'distance_from_center_km', 'avg_route_time_s', 'num_points', 'cluster_fraction',
)

def original_polygon_id(pid, aug_map):
    """Group each rotated/scaled variant with its untransformed polygon."""
    if aug_map[pid].get('augmented', False):
        prefix = 'AUG--AUG-'
        if not pid.startswith(prefix):
            raise ValueError(f'Unknown augmented ID format: {pid}')
        source = 'AUG--' + pid[len(prefix):].rsplit('-', 1)[0]
        if source not in aug_map or aug_map[source].get('augmented', False):
            raise ValueError(f'Cannot find original polygon of {pid}: {source}')
        return source
    return pid

def polygon_features(pid, properties, aug_map):
    prop, record = properties[pid], aug_map[pid]
    dist = np.asarray(record['distance matrix'], dtype=float)
    if dist.ndim != 2 or dist.shape[0] != dist.shape[1] or not np.isfinite(dist).all():
        raise ValueError(f'{pid}: invalid route-time matrix')
    return [
        prop['area_km2'], prop['centroid_lat'], prop['centroid_lon'],
        prop['bounding_box']['width_km'], prop['bounding_box']['height_km'],
        prop['relative_position']['normalized_lat'],
        prop['relative_position']['normalized_lon'],
        prop['relative_position']['distance_from_center_km'],
        float(dist.mean()), prop['num_points'], float(record['cluster']) / 7.0,
    ]

def courier_sample(courier_id, courier, scenario_policy, properties, aug_map,
                   polygon_ids=None):
    ids = list(dict.fromkeys(courier['polygon_list'] if polygon_ids is None else polygon_ids))
    missing = [p for p in ids if p not in scenario_policy or p not in properties or p not in aug_map]
    if missing:
        raise ValueError(f'Courier {courier_id}: {len(missing)} missing labels/features: {missing[:5]}')
    X = np.asarray([polygon_features(p, properties, aug_map) for p in ids], dtype=np.float64)
    y = np.asarray([scenario_policy[p]['policy'] for p in ids], dtype=np.float64)
    groups = np.asarray([original_polygon_id(p, aug_map) for p in ids])
    if X.shape != (len(ids), len(FEATURE_NAMES)) or y.shape != (len(ids), 4):
        raise ValueError(f'Courier {courier_id}: feature or policy shape mismatch')
    if not (np.isfinite(X).all() and np.isfinite(y).all() and
            (y >= 0).all() and np.allclose(y.sum(axis=1), 1, atol=1e-5)):
        raise ValueError(f'Courier {courier_id}: nonfinite feature or invalid policy')
    return ids, X, y, groups

def grouped_splits(groups, seed=42):
    """One fixed train/val/test split reused across all eight scenarios."""
    idx = np.arange(len(groups))
    if len(np.unique(groups)) < 10:
        raise ValueError('Need at least 10 distinct original polygons for grouped splits')
    outer = GroupShuffleSplit(n_splits=1, test_size=.20, random_state=seed)
    dev, test = next(outer.split(idx, groups=groups))
    inner = GroupShuffleSplit(n_splits=1, test_size=.20, random_state=seed+1)
    train_rel, val_rel = next(inner.split(dev, groups=groups[dev]))
    train, val = dev[train_rel], dev[val_rel]
    sets = [set(groups[z]) for z in (train, val, test)]
    assert not (sets[0] & sets[1] or sets[0] & sets[2] or sets[1] & sets[2])
    return train, val, test


In [16]:
def train_scenario_model(X, y, split, output_path, scenario, courier_id,
                         seed=42, max_epochs=500, patience=100):
    train, val, test = split
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    scaler = StandardScaler().fit(X[train])  # no validation/test leakage
    X_train = torch.tensor(scaler.transform(X[train]), dtype=torch.float32)
    X_val = torch.tensor(scaler.transform(X[val]), dtype=torch.float32)
    X_test = torch.tensor(scaler.transform(X[test]), dtype=torch.float32)
    y_train = torch.tensor(y[train], dtype=torch.float32)
    y_val = torch.tensor(y[val], dtype=torch.float32)
    y_test = torch.tensor(y[test], dtype=torch.float32)
    loader = DataLoader(TensorDataset(X_train, y_train),
                        batch_size=min(16, len(train)), shuffle=True)
    model = PolygonAlgNN(input_dim=len(FEATURE_NAMES), hidden_dim=128, classes=4)
    # Match the optimization schedule in the original Supportive NN notebook.
    optimizer = torch.optim.Adam(model.parameters(), lr=0.015, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max_epochs, eta_min=1e-5)
    criterion = nn.KLDivLoss(reduction='batchmean')
    best, best_state, stale = float('inf'), None, 0
    history = {'train_kl': [], 'val_kl': []}
    for epoch in range(max_epochs):
        model.train()
        for bx, by in loader:
            optimizer.zero_grad()
            loss = criterion(F.log_softmax(model(bx), dim=1), by)
            loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = float(criterion(F.log_softmax(model(X_val), dim=1), y_val))
            train_loss = float(criterion(F.log_softmax(model(X_train), dim=1), y_train))
        history['train_kl'].append(train_loss)
        history['val_kl'].append(val_loss)
        scheduler.step()
        smoothed_val = float(np.mean(history['val_kl'][-5:]))
        if smoothed_val < best - 1e-4:
            best, stale = smoothed_val, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            stale += 1
            if stale >= patience and epoch + 1 >= patience:
                break
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        probabilities = F.softmax(model(X_test), dim=1).numpy()
        true = y_test.numpy()
        test_kl = float(criterion(F.log_softmax(model(X_test), dim=1), y_test))
        final_train_kl = float(criterion(F.log_softmax(model(X_train), dim=1), y_train))
        final_val_kl = float(criterion(F.log_softmax(model(X_val), dim=1), y_val))
    # The constant predictor reveals whether geometry adds information beyond
    # the courier's unconditional mean policy.
    constant = np.clip(y[train].mean(axis=0), 1e-8, 1)
    constant /= constant.sum()
    constant_test_kl = float(np.mean(np.sum(
        np.where(true > 0, true * (np.log(np.clip(true, 1e-8, 1))
                                   - np.log(constant)), 0.0), axis=1)))
    metrics = {
        'test_kl': test_kl,
        'constant_test_kl': constant_test_kl,
        'train_kl': final_train_kl, 'final_val_kl': final_val_kl,
        'test_mse': float(np.mean((probabilities - true)**2)),
        'test_top1': float(np.mean(probabilities.argmax(1) == true.argmax(1))),
        'val_kl': best, 'n_train': len(train), 'n_val': len(val),
        'n_test': len(test), 'epochs': epoch+1,
        'target_mean_test': true.mean(axis=0).tolist(),
        'prediction_mean_test': probabilities.mean(axis=0).tolist(),
        'target_std_test': true.std(axis=0).tolist(),
        'prediction_std_test': probabilities.std(axis=0).tolist(),
        'train_label_mean': y[train].mean(axis=0).tolist(),
        'constant_policy': constant.tolist(),
        'mean_policy_l1_gap': float(np.abs(true.mean(axis=0) - probabilities.mean(axis=0)).sum()),
    }
    checkpoint = {
        'model_state_dict': model.state_dict(), 'input_dim': len(FEATURE_NAMES),
        'hidden_dim': 128, 'classes': 4, 'algorithms': list(cfg.algorithms),
        'feature_names': list(FEATURE_NAMES), 'scaler_mean': scaler.mean_,
        'scaler_scale': scaler.scale_, 'scenario': scenario,
        'courier_id': courier_id, 'seed': seed, 'metrics': metrics,
        'loss_history': history,
    }
    torch.save(checkpoint, output_path)
    return metrics

def load_scenario_model(checkpoint_path, device='cpu'):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model = PolygonAlgNN(input_dim=checkpoint['input_dim'],
                         hidden_dim=checkpoint['hidden_dim'], classes=checkpoint['classes'])
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device).eval()
    return model, checkpoint

def predict_policy(model, checkpoint, raw_features, device='cpu'):
    x = np.asarray(raw_features, dtype=float).reshape(-1, checkpoint['input_dim'])
    standardized = (x - checkpoint['scaler_mean']) / checkpoint['scaler_scale']
    with torch.no_grad():
        logits = model(torch.as_tensor(standardized, dtype=torch.float32, device=device))
        return F.softmax(logits, dim=1).cpu().numpy()


In [17]:
# Train eight separate models for each courier. No scenario predictor is added.
metrics_rows = []
cluster_rows = []
for courier_id, courier in trainer.items():
    base = scenario_labels['independent']['gaussian']
    all_ids = list(dict.fromkeys(courier['polygon_list']))
    # Replays only cover polygons with valid triplets. Restrict the comparison
    # to the same polygon set in every scenario and report this coverage.
    shared_ids = [p for p in all_ids if p in properties_global and p in AUG_global_map
                  and all(p in scenario_labels[m][d]
                          for m in rb.SPATIAL_MODES for d in rb.SCENARIOS)]
    print('Courier', courier_id, 'common replay coverage:', len(shared_ids), '/', len(all_ids))
    polygon_ids, X, _, groups = courier_sample(
        courier_id, courier, base, properties_global, AUG_global_map,
        polygon_ids=shared_ids)
    split = grouped_splits(groups, TRAINING_SEED)
    for mode in rb.SPATIAL_MODES:
        for distribution in rb.SCENARIOS:
            policy_map = scenario_labels[mode][distribution]
            if (mode, distribution) == ('spatial', 'gaussian'):
                # Exact target of the published courier checkpoint. Replaying
                # just two calls can produce a different target policy.
                local = courier['EPOS_local_strat']
                absent = [p for p in polygon_ids if p not in local]
                print('Courier', courier_id, 'Gaussian label fallback:',
                      len(absent), '/', len(polygon_ids))
                # The original notebook used the courier's global policy
                # when a polygon-specific policy was unavailable.
                policy_map = {p: {'policy': local.get(p, courier['EPOS_global_strat'])}
                              for p in polygon_ids}
            ids, X_s, y, groups_s = courier_sample(
                courier_id, courier, policy_map,
                properties_global, AUG_global_map, polygon_ids=polygon_ids)
            assert ids == polygon_ids and np.array_equal(groups_s, groups)
            scenario = (mode, distribution)
            name = f'courier_{courier_id}_{mode}_{distribution}.pth'
            path = OUT_DIR / name
            result = train_scenario_model(X_s, y, split, path, scenario, courier_id,
                                          seed=TRAINING_SEED)
            metrics_rows.append({'courier_id': courier_id, 'spatial_mode': mode,
                                 'distribution': distribution, 'checkpoint': str(path), **result})
            trained_model, metadata = load_scenario_model(path)
            test_ids = split[2]
            pred = predict_policy(trained_model, metadata, X_s[test_ids])
            for cluster in sorted({AUG_global_map[ids[i]]['cluster'] for i in test_ids}):
                mask = np.array([AUG_global_map[ids[i]]['cluster'] == cluster for i in test_ids])
                cluster_rows.append({'courier_id': courier_id, 'spatial_mode': mode,
                                     'distribution': distribution, 'cluster': cluster,
                                     'n_test': int(mask.sum()),
                                     'target_mean': y[test_ids][mask].mean(axis=0).tolist(),
                                     'predicted_mean': pred[mask].mean(axis=0).tolist(),
                                     'target_std': y[test_ids][mask].std(axis=0).tolist(),
                                     'predicted_std': pred[mask].std(axis=0).tolist()})
            print(courier_id, scenario, 'test KL', round(result['test_kl'], 4),
                  'constant KL', round(result['constant_test_kl'], 4),
                  'target mean', np.round(result['target_mean_test'], 3),
                  'pred mean', np.round(result['prediction_mean_test'], 3))
metrics_table = pd.DataFrame(metrics_rows)
metrics_table.to_csv(OUT_DIR / 'heldout_metrics.csv', index=False)
cluster_metrics = pd.DataFrame(cluster_rows)
cluster_metrics.to_csv(OUT_DIR / 'heldout_cluster_metrics.csv', index=False)
display(metrics_table)
display(cluster_metrics)


Courier 35 common replay coverage: 3850 / 3850
35 ('independent', 'gaussian') test KL 0.1204 constant KL 0.1934 target mean [0.051 0.036 0.412 0.502] pred mean [0.048 0.04  0.425 0.486]
35 ('independent', 'beta') test KL 0.1137 constant KL 0.1907 target mean [0.049 0.033 0.416 0.502] pred mean [0.047 0.031 0.436 0.487]
35 ('independent', 'dirichlet') test KL 0.1161 constant KL 0.1924 target mean [0.051 0.033 0.414 0.502] pred mean [0.048 0.029 0.43  0.493]
35 ('independent', 'ordinal_posterior') test KL 0.204 constant KL 0.2262 target mean [0.104 0.102 0.352 0.442] pred mean [0.112 0.083 0.332 0.474]
Courier 35 Gaussian label fallback: 0 / 3850
35 ('spatial', 'gaussian') test KL 0.1242 constant KL 0.1934 target mean [0.053 0.036 0.404 0.507] pred mean [0.049 0.026 0.376 0.549]
35 ('spatial', 'beta') test KL 0.1124 constant KL 0.188 target mean [0.051 0.033 0.409 0.507] pred mean [0.051 0.032 0.401 0.516]
35 ('spatial', 'dirichlet') test KL 0.1147 constant KL 0.1894 target mean [0.052 0

,courier_id,spatial_mode,distribution,checkpoint,test_kl,constant_test_kl,train_kl,final_val_kl,test_mse,test_top1,...,n_val,n_test,epochs,target_mean_test,prediction_mean_test,target_std_test,prediction_std_test,train_label_mean,constant_policy,mean_policy_l1_gap
0,35,independent,gaussian,courier_models_fd_8_scenarios\courier_35_indep...,0.120404,0.193389,0.141008,0.150138,0.013226,0.640260,...,620,770,149,"[0.050594761967659, 0.03596550598740578, 0.411...","[0.04782382398843765, 0.04045872390270233, 0.4...","[0.08994712680578232, 0.0873568132519722, 0.15...","[0.05212969705462456, 0.05978242680430412, 0.0...","[0.05207361322719734, 0.03937362819939831, 0.4...","[0.05207361322719732, 0.03937362819939829, 0.4...",0.035819
1,35,independent,beta,courier_models_fd_8_scenarios\courier_35_indep...,0.113746,0.190696,0.121250,0.131311,0.013750,0.635065,...,620,770,199,"[0.04875080659985542, 0.032823748886585236, 0....","[0.046645160764455795, 0.03053942136466503, 0....","[0.08870764076709747, 0.084384024143219, 0.152...","[0.04879993572831154, 0.05227097123861313, 0.0...","[0.04909363251434609, 0.03593373470868947, 0.4...","[0.04909363251434606, 0.03593373470868945, 0.4...",0.040411
2,35,independent,dirichlet,courier_models_fd_8_scenarios\courier_35_indep...,0.116091,0.192385,0.130614,0.133926,0.013675,0.657143,...,620,770,110,"[0.05057135596871376, 0.03286196291446686, 0.4...","[0.04801338165998459, 0.029482517391443253, 0....","[0.08884628117084503, 0.0843699499964714, 0.15...","[0.04364186152815819, 0.048100054264068604, 0....","[0.05024823518503452, 0.036045116674419085, 0....","[0.0502482351850345, 0.03604511667441907, 0.41...",0.030908
3,35,independent,ordinal_posterior,courier_models_fd_8_scenarios\courier_35_indep...,0.204026,0.226243,0.209833,0.207838,0.018068,0.576623,...,620,770,200,"[0.10386692732572556, 0.10180921852588654, 0.3...","[0.11155403405427933, 0.08299065381288528, 0.3...","[0.11011425405740738, 0.12010776251554489, 0.1...","[0.052344564348459244, 0.03422597050666809, 0....","[0.09582481031163599, 0.10179074157777801, 0.3...","[0.09582481031163595, 0.10179074157777797, 0.3...",0.078708
4,35,spatial,gaussian,courier_models_fd_8_scenarios\courier_35_spati...,0.124211,0.193357,0.131534,0.145118,0.014377,0.710390,...,620,770,203,"[0.05269206687808037, 0.03598032519221306, 0.4...","[0.049083225429058075, 0.02593083865940571, 0....","[0.08970444649457932, 0.08808444440364838, 0.1...","[0.043343156576156616, 0.04109615460038185, 0....","[0.05135782525720913, 0.038803129321437646, 0....","[0.05135782525720908, 0.03880312932143761, 0.4...",0.083842
5,35,spatial,beta,courier_models_fd_8_scenarios\courier_35_spati...,0.112443,0.187977,0.119097,0.134927,0.013156,0.703896,...,620,770,145,"[0.050563327968120575, 0.0328572653234005, 0.4...","[0.050568029284477234, 0.03202896937727928, 0....","[0.08814956992864609, 0.08437234163284302, 0.1...","[0.05343250930309296, 0.057544101029634476, 0....","[0.0508922736710402, 0.03597601448463402, 0.40...","[0.05089227367104022, 0.035976014484634035, 0....",0.017516
6,35,spatial,dirichlet,courier_models_fd_8_scenarios\courier_35_spati...,0.114666,0.189401,0.121017,0.132087,0.014104,0.703896,...,620,770,229,"[0.05234082043170929, 0.032928466796875, 0.409...","[0.06036942079663277, 0.038722164928913116, 0....","[0.08832214772701263, 0.08435066044330597, 0.1...","[0.0659589096903801, 0.07394734025001526, 0.06...","[0.05232359319788201, 0.03604802587300102, 0.4...","[0.05232359319788203, 0.03604802587300104, 0.4...",0.038806
7,35,spatial,ordinal_posterior,courier_models_fd_8_scenarios\courier_35_spati...,0.196237,0.230514,0.198094,0.200293,0.018016,0.580519,...,620,770,302,"[0.1046726405620575, 0.09844842553138733, 0.35...","[0.09528739005327225, 0.10497547686100006, 0.3...","[0.1102413460612297, 0.12006683647632599, 0.14...","[0.04595211148262024, 0.054841190576553345, 0....","[0.09760177267484106, 0.10295564279122914, 0.3...","[0.09760177267484106, 0.10295564279122914, 0.3...",0.035777


,courier_id,spatial_mode,distribution,cluster,n_test,target_mean,predicted_mean,target_std,predicted_std
0,35,independent,gaussian,0,56,"[0.14732142857142858, 0.15368152258181622, 0.3...","[0.1294034868478775, 0.13789941370487213, 0.34...","[0.12299086887460633, 0.12331158661575113, 0.1...","[0.049302637577056885, 0.049269337207078934, 0..."
1,35,independent,gaussian,2,501,"[0.026570451014123725, 0.0034521818861808487, ...","[0.020330457016825676, 0.0052874088287353516, ...","[0.055998755737227096, 0.025938202411333083, 0...","[0.015022249892354012, 0.009279442019760609, 0..."
2,35,independent,gaussian,3,6,"[0.20833333333333334, 0.20833333333457035, 0.3...","[0.1494283676147461, 0.151645690202713, 0.3232...","[0.09316949906249124, 0.09316949905972523, 0.2...","[0.00841299444437027, 0.008326583541929722, 0...."
3,35,independent,gaussian,5,147,"[0.06390416071560477, 0.05830768722439944, 0.3...","[0.07834786921739578, 0.08399534225463867, 0.3...","[0.1047567058976578, 0.10524917471767425, 0.13...","[0.05811350792646408, 0.06019563600420952, 0.0..."
4,35,independent,gaussian,7,60,"[0.11253775828407105, 0.12560836582342932, 0.3...","[0.1163078248500824, 0.12541115283966064, 0.35...","[0.12433960954449289, 0.12698907753353433, 0.1...","[0.043728116899728775, 0.04133440554141998, 0...."
5,35,independent,beta,0,56,"[0.14732142857142858, 0.1473219064523101, 0.35...","[0.12365270406007767, 0.12368040531873703, 0.3...","[0.12299086887460633, 0.12299029649775532, 0.2...","[0.0352933295071125, 0.036347754299640656, 0.0..."
6,35,independent,beta,2,501,"[0.02422692415530183, 0.00104157724595464, 0.4...","[0.0246846005320549, 0.0016676582163199782, 0....","[0.05208138940829751, 0.015762445270331143, 0....","[0.025499414652585983, 0.007919827476143837, 0..."
7,35,independent,beta,3,6,"[0.20833333333333334, 0.20833333333333334, 0.3...","[0.1389089971780777, 0.1378144770860672, 0.315...","[0.09316949906249124, 0.09316949906249124, 0.1...","[0.0022898614406585693, 0.003912582993507385, ..."
8,35,independent,beta,5,147,"[0.062244119995819865, 0.057838119126421776, 0...","[0.06425242871046066, 0.05861300602555275, 0.4...","[0.10446359738518189, 0.10540653714747322, 0.1...","[0.05514382943511009, 0.056117165833711624, 0...."
9,35,independent,beta,7,60,"[0.11250916779308767, 0.11250386305933432, 0.3...","[0.10577765107154846, 0.10517916828393936, 0.3...","[0.12436515671767814, 0.12436993681118694, 0.1...","[0.04301203042268753, 0.043360378593206406, 0...."


In [18]:
# Compare against the courier checkpoint's spatial/Gaussian target.
# Independent/Gaussian trains first but is a different experiment.
first = metrics_table.loc[
    (metrics_table['spatial_mode'] == 'spatial') &
    (metrics_table['distribution'] == 'gaussian')].iloc[0]
model, metadata = load_scenario_model(first['checkpoint'])
courier_id = first['courier_id']
ids, X, y, groups = courier_sample(
    courier_id, trainer[courier_id],
    ({p: {'policy': trainer[courier_id]['EPOS_local_strat'].get(
                     p, trainer[courier_id]['EPOS_global_strat'])}
      for p in trainer[courier_id]['polygon_list']}
     if (first['spatial_mode'], first['distribution']) == ('spatial', 'gaussian')
     else scenario_labels[first['spatial_mode']][first['distribution']]),
    properties_global, AUG_global_map,
    polygon_ids=[p for p in dict.fromkeys(trainer[courier_id]['polygon_list'])
                 if p in properties_global and p in AUG_global_map
                 and all(p in scenario_labels[m][d]
                         for m in rb.SPATIAL_MODES for d in rb.SCENARIOS)])
_, _, test_idx = grouped_splits(groups, TRAINING_SEED)
print('Held-out polygon:', ids[test_idx[0]],
      'predicted:', predict_policy(model, metadata, X[test_idx[0]])[0],
      'target:', y[test_idx[0]])
print('Same held-out set mean target:', np.round(first['target_mean_test'], 4))
print('Same held-out set mean prediction:', np.round(first['prediction_mean_test'], 4))
print('Train-only mean policy:', np.round(first['train_label_mean'], 4))


Held-out polygon: AUG--AUG-G-236-1-374-5 predicted: [0.11046048 0.09367093 0.3449881  0.4508805 ] target: [1.82865866e-37 2.25421130e-19 5.97961216e-02 9.40203878e-01]
Same held-out set mean target: [0.0527 0.036  0.4039 0.5074]
Same held-out set mean prediction: [0.0491 0.0259 0.3756 0.5493]
Train-only mean policy: [0.0514 0.0388 0.4062 0.5037]
